# AM5061 · Week 8 · Waste-heat recovery boiler

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

A **two-pass fire-tube waste-heat recovery boiler** on a cement kiln, taken
directly from the HEX Class Discussion deck, Case 2.

| | |
|---|---|
| flue gas in | 400 °C, C_h = 315 W/K |
| pass 1 | **one** tube, D₁ = 20 cm, L₁ = 4 m, U₁ = 55 W/m²K |
| pass 2 | **ten** tubes, D₂ = 6 cm, L₂ = ?, U₂ = 110 W/m²K |
| shell | boiling water, T_sat = 150 °C, constant throughout |
| gas out | 200 °C |

Deliverable **D-8**: size pass 2. Then answer **D5**: why is U₂ twice U₁?

### Why this case is easy, and why that matters

The shell side is boiling at constant temperature, so `C_max → ∞`, the capacity
ratio `C_r = 0`, and **every configuration collapses to the same relation**:

`ε = 1 − exp(−NTU)`

Counter-flow, parallel-flow, cross-flow, shell-and-tube — identical. Phase
change removes the configuration question entirely.


## 1. Setup, and the C_r = 0 collapse

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
am.style_plots()
PI = np.pi

C_h     = 315.0        # W/K   gas capacity rate
T_g_in  = 400.0        # C
T_sat   = 150.0        # C     boiling shell side
T_g_out = 200.0        # C     target stack temperature

D1, L1, N1, U1 = 0.20, 4.0, 1,  55.0
D2,     N2, U2 = 0.06,      10, 110.0

print("  Every configuration gives the same effectiveness when C_r = 0:")
for cfg in ("counter", "parallel", "shell1", "cross-both-unmixed"):
    print(f"    {cfg:20s} eps(NTU=1) = {am.effectiveness(cfg, 1.0, 0.0):.6f}")
print(f"    closed form 1-exp(-1)      = {1-np.exp(-1):.6f}")


## 2. Pass 1, as built

In [ ]:
A1   = PI*D1*L1*N1
NTU1 = U1*A1/C_h
eps1 = 1 - np.exp(-NTU1)
Q1   = eps1*C_h*(T_g_in - T_sat)
T_m  = T_g_in - Q1/C_h                    # gas temperature between passes

print(f"  A1    = pi*{D1}*{L1}        = {A1:.4f} m2")
print(f"  NTU1  = U1*A1/C_h            = {NTU1:.4f}")
print(f"  eps1  = 1-exp(-NTU1)         = {eps1:.4f}")
print(f"  Q1    = eps1*C_h*(400-150)   = {Q1/1e3:.2f} kW")
print(f"  T_m   = 400 - Q1/C_h         = {T_m:.1f} C")


## 3. Pass 2 — the deliverable

In [ ]:
Q_total = C_h*(T_g_in - T_g_out)
Q2      = Q_total - Q1
eps2    = Q2/(C_h*(T_m - T_sat))
NTU2    = -np.log(1 - eps2)
A2      = NTU2*C_h/U2
L2      = A2/(N2*PI*D2)

print(f"  Q_total = C_h*(400-200)      = {Q_total/1e3:.2f} kW")
print(f"  Q2      = Q_total - Q1       = {Q2/1e3:.2f} kW")
print(f"  eps2    = Q2/[C_h*(T_m-150)] = {eps2:.4f}")
print(f"  NTU2    = -ln(1-eps2)        = {NTU2:.4f}")
print(f"  A2      = NTU2*C_h/U2        = {A2:.4f} m2")
print(f"  L2      = A2/(10*pi*0.06)    = {L2:.4f} m")
print(f"\n  gas path: 400 C -> {T_m:.0f} C (pass 1) -> {T_g_out:.0f} C (pass 2)")


> **Check.** L₂ = **1.78 m**, matching the HEX deck.

### Steam production

The feedwater has to be heated to saturation *and* evaporated. The sensible
part is easy to forget and it is not small.


In [ ]:
T_fw   = 50.0                                    # C, feedwater
cp_w   = PropsSI("C","P",4.76e5,"Q",0,"Water")   # ~150 C saturation
h_fg   = (PropsSI("H","P",4.76e5,"Q",1,"Water")
          - PropsSI("H","P",4.76e5,"Q",0,"Water"))
h_per_kg = cp_w*(T_sat - T_fw) + h_fg
m_steam  = Q_total/h_per_kg
print(f"  sensible  {cp_w*(T_sat-T_fw)/1e3:8.1f} kJ/kg   ({100*cp_w*(T_sat-T_fw)/h_per_kg:.1f}% of the total)")
print(f"  latent    {h_fg/1e3:8.1f} kJ/kg")
print(f"  total     {h_per_kg/1e3:8.1f} kJ/kg")
print(f"  steam     {m_steam:8.4f} kg/s = {m_steam*3600:.1f} kg/h")


## 4. D5 — why is U₂ twice U₁?

The deck's hint is that ten small tubes raise the gas velocity relative to one
large tube. **Check it quantitatively rather than accepting it**, which is what
the question actually asks.


In [ ]:
A_flow_1 = N1*PI*D1**2/4
A_flow_2 = N2*PI*D2**2/4
v_ratio  = A_flow_1/A_flow_2

# Dittus-Boelter:  h ~ Re^0.8 / D  ~  (V*D)^0.8 / D  =  V^0.8 * D^-0.2
h_ratio = v_ratio**0.8 * (D2/D1)**-0.2

print(f"  flow area, pass 1 (1 x 20 cm)   {A_flow_1:.6f} m2")
print(f"  flow area, pass 2 (10 x 6 cm)   {A_flow_2:.6f} m2")
print(f"  velocity ratio  V2/V1           {v_ratio:.4f}")
print(f"  diameter ratio  D2/D1           {D2/D1:.4f}")
print(f"\n  h2/h1 from Dittus-Boelter      {h_ratio:.4f}")
print(f"  U2/U1 as stipulated             {U2/U1:.4f}")
print(f"\n  *** These do not agree. ***")


### The honest answer

The velocity only rises by **11%**, because ten 6 cm tubes have almost the same
total flow area as one 20 cm tube. Working the Dittus–Boelter scaling through
gives **h₂/h₁ ≈ 1.38**, not 2.0.

So the stipulated `U₂ = 110 W/m²K` is a **design value, not a derived one**.
Most of the gain comes from the velocity being higher *and* the tube being
narrower; the rest is whatever the original designer assumed about fouling,
entry effects and the shell side.

**This is the point of the question.** The instinct — more small tubes means
higher velocity means higher U — is right in direction. The factor of two is
not something the geometry alone delivers, and a designer who assumes it
without checking will undersize pass 2 by about a third.

Report the number you compute, state the assumption you are testing, and say
where the rest would have to come from. That is a better answer than
reproducing 2.0.


In [ ]:
# What if U2 really were only 1.38 x U1?
U2_derived = U1*h_ratio
A2_d = NTU2*C_h/U2_derived
L2_d = A2_d/(N2*PI*D2)
print(f"  with the stipulated U2 = {U2:.0f} W/m2K  ->  L2 = {L2:.3f} m")
print(f"  with a derived     U2 = {U2_derived:.1f} W/m2K  ->  L2 = {L2_d:.3f} m")
print(f"  {100*(L2_d-L2)/L2:+.0f}% more tube, on the same duty.")


## 5. The stack temperature is the real decision

In [ ]:
def pass2_length(T_out, U=U2):
    Q_t = C_h*(T_g_in - T_out)
    Q_2 = Q_t - Q1
    if Q_2 <= 0: return np.nan, Q_2/1e3
    e2  = Q_2/(C_h*(T_m - T_sat))
    if e2 >= 1: return np.nan, Q_2/1e3
    return (-np.log(1-e2))*C_h/U/(N2*PI*D2), Q_2/1e3

rows = []
for T_out in np.arange(170.0, 261.0, 5.0):
    L, q2 = pass2_length(float(T_out))
    rows.append({"T_stack, C": float(T_out), "Q2, kW": q2, "L2, m": L,
                 "Q_total, kW": C_h*(T_g_in-T_out)/1e3,
                 "steam, kg/h": C_h*(T_g_in-T_out)/h_per_kg*3600})

fig, ax = plt.subplots()
ax.plot([r_["T_stack, C"] for r_ in rows], [r_["L2, m"] for r_ in rows],
        "o-", lw=2.4, color=am.ORANGE)
ax.axvline(170, color="#B03A2E", ls="--")
ax.text(172, 4, "acid dew point risk\nbelow ~170 C", color="#B03A2E", fontsize=9)
ax.plot(T_g_out, L2, "o", ms=11, color=am.NAVY, zorder=5)
ax.annotate(f"  design point\n  {L2:.2f} m", (T_g_out, L2), color=am.NAVY, fontsize=10)
ax.set_xlabel("stack temperature  (°C)"); ax.set_ylabel("pass 2 length  (m)")
ax.set_title("The last kelvin of recovery is the expensive one")
plt.tight_layout(); plt.show()
print("  Thermodynamics caps recovery at T_sat = 150 C. Acid dew point sets a")
print("  higher practical floor. Between them sits an economic optimum, and")
print("  that is a Week 14 calculation, not a Week 8 one.")


## 6. The deliverable

In [ ]:
design = [{"pass": 1, "tubes": N1, "D, m": D1, "L, m": L1, "U, W/m2K": U1,
           "A, m2": A1, "NTU": NTU1, "eps": eps1, "Q, kW": Q1/1e3,
           "T_gas_in, C": T_g_in, "T_gas_out, C": T_m},
          {"pass": 2, "tubes": N2, "D, m": D2, "L, m": L2, "U, W/m2K": U2,
           "A, m2": A2, "NTU": NTU2, "eps": eps2, "Q, kW": Q2/1e3,
           "T_gas_in, C": T_m, "T_gas_out, C": T_g_out}]

path = am.to_excel("AM5061_D8_WHRBoiler.xlsx",
    {"Design": design, "Stack temperature sweep": rows},
    title="AM5061 D-8 . Cement kiln waste-heat recovery boiler",
    summary=[("Gas capacity rate C_h", C_h, "W/K"),
             ("Gas inlet", T_g_in, "C"), ("Steam saturation", T_sat, "C"),
             ("Capacity ratio C_r", 0.0, "-"),
             ("Gas between passes T_m", T_m, "C"),
             ("Pass 2 length", L2, "m"),
             ("Total duty", Q_total/1e3, "kW"),
             ("Steam production", m_steam*3600, "kg/h"),
             ("U2/U1 stipulated", U2/U1, "-"),
             ("U2/U1 from Dittus-Boelter scaling", h_ratio, "-")],
    sources=[("Case data", "AM5061 HEX Class Discussion deck, Case 2, slides 11-13"),
             ("Effectiveness", "C_r = 0 limit: eps = 1 - exp(-NTU), all configurations"),
             ("Water/steam", "CoolProp, IAPWS-95, saturation at 150 C"),
             ("U1, U2", "STIPULATED in the source deck; see D5 discussion")])
print("written:", path)


## What to hand in

1. Pass 2 length, with the ε-NTU working shown.
2. Steam production, and the sensible fraction of the feedwater load.
3. **D5 answered honestly**: the velocity ratio, the Dittus–Boelter scaling,
   the number you get, and why it differs from the stipulated 2.0.
4. The stack-temperature curve, with your recommended value and its
   justification.
5. The workbook.

**One paragraph:** `C_r = 0` because the shell side boils. At what point in
this design would that stop being true, and what would change if it did?
